# Lookup: shape + color without retraining
Two "models", both lookups in a reference gallery (`data/gallery.npz`):
- **Shape**: a color-blind fingerprint (ResNet18 on grayscale crops) → nearest reference photos → `part_num`.
- **Color**: median Lab color of the brick pixels → nearest reference photos → LEGO color ID.

**Adding a new piece = adding ~20 photos to the gallery** (`lego_lookup.add_photos` or `scripts/add_piece.py`). No training.

To prove that, 3 shapes are **held out**: the fingerprint model never trains on them. They only get gallery photos, exactly like a future new piece.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')   # all paths below are from the repo root
import csv, sys, time, random
from collections import defaultdict, Counter
import cv2, numpy as np, torch, torchvision, matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
sys.path.insert(0, '.'); import importlib, lego_lookup as L; importlib.reload(L)

lab = {r['path']: r for r in csv.DictReader(open('data/labels.csv'))}
boxes = {r['path'] for r in csv.DictReader(open('data/boxes.csv'))}
ranges = [(int(r['start']), int(r['end'])) for r in csv.DictReader(open('data/ranges.csv'))]
run_of = lambda p: next(i for i, (a, b) in enumerate(ranges) if a <= int(os.path.basename(p)[4:8]) <= b)
split, runs = {}, defaultdict(list)   # same per-run 70/15/15 split as compare.ipynb
for p in boxes: runs[run_of(p)].append(p)
for ps in runs.values():
    ps.sort()
    for i, p in enumerate(ps): split[p] = 'train' if i < 0.7 * len(ps) else 'val' if i < 0.85 * len(ps) else 'test'
P = {k: sorted(p for p in boxes if split[p] == k) for k in ('train', 'val', 'test')}
crop = lambda p: cv2.cvtColor(cv2.imread('crops224/' + p), cv2.COLOR_BGR2RGB)

HOLD = ['15254', '92950', '3062b']   # "future new pieces": never seen by the fingerprint model
SHAPES = sorted({r['part_num'] for r in lab.values()} - set(HOLD)); SI = {s: i for i, s in enumerate(SHAPES)}
print(len(SHAPES), 'trained shapes +', len(HOLD), 'held out |', {k: len(v) for k, v in P.items()})

## 1. Train the color-blind shape fingerprint (~3 min, skipped if `models/shape_embed.pt` exists)

In [ ]:
class Crops(Dataset):
    def __init__(s, ps, tf): s.ps, s.tf = [p for p in ps if lab[p]['part_num'] in SI], tf
    def __len__(s): return len(s.ps)
    def __getitem__(s, i):
        p = s.ps[i]
        return s.tf(torch.from_numpy(L.gray3(crop(p))).permute(2, 0, 1)), SI[lab[p]['part_num']]
norm = T.Compose([T.ToDtype(torch.float32, scale=True), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train_tf = T.Compose([T.RandomRotation(180), T.RandomResizedCrop(224, scale=(0.6, 1)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.4, 0.4), norm])   # grayscale input: brightness/contrast only, color can't leak

if not os.path.exists(L.EMBED_PT):
    m = torchvision.models.resnet18(weights='DEFAULT'); m.fc = torch.nn.Linear(512, len(SHAPES)); m = m.to(L.DEV)
    tl = DataLoader(Crops(P['train'], train_tf), batch_size=64, shuffle=True, num_workers=8)
    vl = DataLoader(Crops(P['val'], norm), batch_size=128, num_workers=8)
    EP = 10; opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=EP * len(tl)); scaler = torch.amp.GradScaler()
    for ep in range(EP):
        m.train(); t = time.time()
        for x, y in tl:
            with torch.autocast('cuda'): loss = torch.nn.functional.cross_entropy(m(x.to(L.DEV)), y.to(L.DEV), label_smoothing=0.1)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        m.eval(); ok = n = 0
        with torch.no_grad():
            for x, y in vl: ok += (m(x.to(L.DEV)).argmax(1).cpu() == y).sum().item(); n += len(y)
        print(f'epoch {ep}: val shape acc {ok / n:.3f} ({time.time() - t:.0f}s)')
    os.makedirs('models', exist_ok=True); torch.save({k: v for k, v in m.state_dict().items() if not k.startswith('fc.')}, L.EMBED_PT)
model = L.load_embedder()

## 2. Build the gallery (the reference sheet)
20 evenly spaced **train** photos per element (shape + color pair), held-out shapes included, as if just added.

In [ ]:
PER = 20
def build(model, ps_by_elem):
    g = L.load_gallery('/nonexistent')
    for e, ps in sorted(ps_by_elem.items()):
        cs = [crop(p) for p in ps]
        g = L.add_entries(g, L.embed(model, cs), np.array([L.brick_lab(c) for c in cs]), lab[ps[0]]['part_num'], lab[ps[0]]['lego_color_id'], ps)
    return g

by_elem = defaultdict(list)
for p in P['train']: by_elem[lab[p]['element_id']].append(p)
pick = {e: [ps[int(i)] for i in np.linspace(0, len(ps) - 1, PER)] for e, ps in by_elem.items()}
t = time.time(); g = build(model, pick); L.save_gallery(g)
print(len(g['vec']), 'gallery entries,', len(set(g['shape'])), 'shapes,', len(set(g['color'])), f'colors -> {L.GALLERY} ({os.path.getsize(L.GALLERY) // 1024} KB, {time.time() - t:.0f}s)')

## 3. Test split: how often is the lookup right?

In [ ]:
T_ps = P['test']
tc = [crop(p) for p in T_ps]
tv, tl_ = L.embed(model, tc), np.array([L.brick_lab(c) for c in tc])

def score(g, vecs, labs, ps, title):
    r = L.recognize(g, vecs, labs)
    sh = np.array([a == lab[p]['part_num'] for (a, _, _, _), p in zip(r, ps)])
    co = np.array([c == lab[p]['lego_color_id'] for (_, _, c, _), p in zip(r, ps)])
    held = np.array([lab[p]['part_num'] in HOLD for p in ps])
    print(f'{title}: shape {sh.mean():.3f} (held-out shapes {sh[held].mean():.3f}) | color {co.mean():.3f} | both {(sh & co).mean():.3f}')
    return r, sh, co

r, sh, co = score(g, tv, tl_, T_ps, 'color-blind fingerprint')
plain = L.load_embedder('/nonexistent')   # ImageNet only, never saw a brick
g0 = build(plain, pick); score(g0, L.embed(plain, tc), tl_, T_ps, 'plain ImageNet fingerprint')
print('shape mix-ups:', Counter((lab[p]['part_num'], a) for (a, *_), p, ok in zip(r, T_ps, sh) if not ok).most_common(6))
print('color mix-ups:', Counter((lab[p]['lego_color_name'], c) for (_, _, c, _), p, ok in zip(r, T_ps, co) if not ok).most_common(6))

## 4. Unknown pieces
A piece that is **not** in the gallery should come out as `unknown`, not as a wrong guess.
Remove each held-out shape from the gallery in turn and compare its best-match similarity with that of known pieces.

In [ ]:
known, unk = [], []
for h in HOLD:
    gk = {k: v[g['shape'] != h] for k, v in g.items()}
    for (_, s, _, _), p in zip(L.recognize(gk, tv, tl_), T_ps):
        (unk if lab[p]['part_num'] == h else known).append(s)
known, unk = np.array(known), np.array(unk)
th = float(np.percentile(known, 5))   # keep 95% of known pieces
print(f'threshold {th:.3f}: known kept {(known >= th).mean():.2f}, unknown caught {(unk < th).mean():.2f}')
plt.hist(known, 50, alpha=.6, density=True, label='piece in gallery'); plt.hist(unk, 50, alpha=.6, density=True, label='piece NOT in gallery')
plt.axvline(th, color='k'); plt.xlabel('best-match similarity'); plt.legend(); plt.show()
print(f'-> set SHAPE_MIN_SIM = {th:.2f} in lego_lookup.py')

## 5. How to add a new piece
Photograph it alone on a table, ~20 photos from different sides, then:
```
python scripts/add_piece.py photos/my_new_piece/ --shape 3001 --color 21
```
or in Python: `L.add_photos(glob.glob('photos/my_new_piece/*.jpg'), shape='3001', color='21')`.
The box detector finds the brick in each photo, crops it, and adds it to `data/gallery.npz`. Commit that file to share the new piece.
Same command for a known shape in a **new color**.